**Zomato Data Analysis Using Python**

Project Overview

This project focuses on cleaning, preprocessing, and exploring Zomato restaurant data using Python. The objective is to understand restaurant data quality, analyze restaurant attributes, and discover useful patterns through Exploratory Data Analysis (EDA).

In [ ]:
# Cell 1 — Imports and plotting setup
from pathlib import Path
import zipfile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

# Your downloaded dataset path (folder or file path)
DATA_LOCATION = Path(r"C:\Users\Ananya\Downloads\archive (1)")

# Outputs will be saved beside the notebook in this folder
OUTPUT_DIR = Path("outputs")
PLOTS_DIR = OUTPUT_DIR / "plots"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset location:", DATA_LOCATION)
print("Output folder:", OUTPUT_DIR.resolve())


In [ ]:
# Cell 2 — Locate and load the real dataset (no fake-data fallback)

def locate_dataset(location: Path) -> Path:
    """Accept a dataset file path or a folder containing CSV/Excel/ZIP files."""
    location = Path(location)

    if location.is_file():
        return location

    # The path may have been supplied without the file extension
    for suffix in (".csv", ".zip", ".xlsx", ".xls"):
        candidate = Path(str(location) + suffix)
        if candidate.is_file():
            return candidate

    if location.is_dir():
        supported = []
        for pattern in ("*.csv", "*.zip", "*.xlsx", "*.xls"):
            supported.extend(location.rglob(pattern))
        supported = sorted(set(supported))

        if len(supported) == 1:
            return supported[0]
        if len(supported) > 1:
            print("Multiple dataset files found:")
            for i, item in enumerate(supported, start=1):
                print(f"  {i}. {item}")
            raise FileNotFoundError(
                "More than one dataset file was found. Set DATA_LOCATION to the exact file path."
            )

    raise FileNotFoundError(
        f"Dataset not found at: {location}\n"
        "Check the path and set DATA_LOCATION to the exact CSV file, or to a folder containing one dataset file."
    )


def load_dataset(path: Path) -> pd.DataFrame:
    """Load CSV, Excel, or ZIP containing a CSV file."""
    suffix = path.suffix.lower()

    if suffix == ".csv":
        last_error = None
        for encoding in ("utf-8", "utf-8-sig", "latin-1"):
            try:
                return pd.read_csv(path, encoding=encoding, low_memory=False)
            except UnicodeDecodeError as exc:
                last_error = exc
        raise ValueError(f"Could not decode CSV file: {path}") from last_error

    if suffix in (".xlsx", ".xls"):
        return pd.read_excel(path)

    if suffix == ".zip":
        with zipfile.ZipFile(path) as zf:
            csv_members = [name for name in zf.namelist() if name.lower().endswith(".csv")]
            if not csv_members:
                raise ValueError("ZIP file does not contain a CSV file.")
            if len(csv_members) > 1:
                print("CSV files inside ZIP:", csv_members)
                raise ValueError("ZIP contains multiple CSV files. Extract it and point DATA_LOCATION to the intended CSV.")
            with zf.open(csv_members[0]) as f:
                return pd.read_csv(f, low_memory=False)

    raise ValueError(f"Unsupported file type '{suffix}'. Use CSV, ZIP containing one CSV, or Excel.")


DATA_FILE = locate_dataset(DATA_LOCATION)
df = load_dataset(DATA_FILE)

# Standardize column headers only (preserve their meaning)
df.columns = df.columns.astype(str).str.strip()

print(f"Loaded file: {DATA_FILE}")
print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("\nColumns:")
print(df.columns.tolist())
display(df.head())


NameError: name 'DATA_LOCATION' is not defined

In [ ]:
 #Cell 3 — Initial profile
print("Data types:")
display(df.dtypes.rename("dtype").to_frame())

print("\nMissing values (top 20):")
missing_before = df.isna().sum().sort_values(ascending=False)
display(missing_before.head(20).rename("missing_count").to_frame())

print("\nExact duplicate rows:", int(df.duplicated().sum()))
print("\nNumeric summary:")
display(df.describe(include="number").T)


In [ ]:
# Cell 4 — Robust column detection and cleaning

def find_column(columns, candidates):
    """Return the first matching column, ignoring case and surrounding spaces."""
    lookup = {str(c).strip().lower(): c for c in columns}
    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]
    return None


def clean_data(data: pd.DataFrame) -> pd.DataFrame:
    """Clean common Zomato columns without modifying the input DataFrame."""
    cleaned = data.copy()
    cleaned.columns = cleaned.columns.astype(str).str.strip()

    # Detect common column variants
    name_col = find_column(cleaned.columns, ["name", "restaurant_name", "restaurant"])
    address_col = find_column(cleaned.columns, ["address", "restaurant_address"])
    rate_col = find_column(cleaned.columns, ["rate", "rating", "aggregate_rating"])
    cost_col = find_column(cleaned.columns, [
        "approx_cost(for two people)", "approx_cost(for two)", "cost_for_two",
        "average_cost_for_two", "cost"
    ])
    votes_col = find_column(cleaned.columns, ["votes", "vote_count", "reviews_count"])
    online_col = find_column(cleaned.columns, ["online_order", "online ordering", "has_online_order"])
    table_col = find_column(cleaned.columns, ["book_table", "table_booking", "has_table_booking"])
    location_col = find_column(cleaned.columns, ["location", "area", "locality"])
    cuisine_col = find_column(cleaned.columns, ["cuisines", "cuisine", "cuisine_type"])

    # Normalize common names for downstream analysis while retaining other source columns
    rename_map = {}
    for original, standard in [
        (name_col, "name"), (address_col, "address"), (rate_col, "rate"),
        (cost_col, "cost_for_two"), (votes_col, "votes"),
        (online_col, "online_order"), (table_col, "book_table"),
        (location_col, "location"), (cuisine_col, "cuisines")
    ]:
        if original is not None and original != standard:
            # Avoid overwriting a separate column that already uses the standardized name
            if standard not in cleaned.columns or standard == original:
                rename_map[original] = standard
    cleaned = cleaned.rename(columns=rename_map)

    # Remove exact duplicate rows
    exact_before = len(cleaned)
    cleaned = cleaned.drop_duplicates().copy()
    print(f"Exact duplicate rows removed: {exact_before - len(cleaned):,}")

    # Remove repeated restaurant name/address combinations if both columns exist
    if {"name", "address"}.issubset(cleaned.columns):
        pair_before = len(cleaned)
        cleaned = cleaned.drop_duplicates(subset=["name", "address"]).copy()
        print(f"Duplicate name + address rows removed: {pair_before - len(cleaned):,}")

    # Ratings like '4.1/5', 'NEW', '-', 'nan'
    if "rate" in cleaned.columns:
        raw_rate = cleaned["rate"].astype("string").str.strip()
        raw_rate = raw_rate.str.replace("/5", "", regex=False)
        raw_rate = raw_rate.replace({
            "NEW": pd.NA, "New": pd.NA, "-": pd.NA, "": pd.NA, "nan": pd.NA
        })
        cleaned["rate"] = pd.to_numeric(raw_rate, errors="coerce")
        missing_ratings = int(cleaned["rate"].isna().sum())
        invalid_ratings = int((~cleaned["rate"].between(0, 5) & cleaned["rate"].notna()).sum())
        print(f"Rows with missing/unusable ratings before rating filter: {missing_ratings:,}")
        print(f"Ratings outside 0–5: {invalid_ratings:,}")
        cleaned.loc[~cleaned["rate"].between(0, 5), "rate"] = np.nan

    # Costs may be strings like '1,200'
    if "cost_for_two" in cleaned.columns:
        cleaned["cost_for_two"] = (
            cleaned["cost_for_two"].astype("string")
            .str.replace(",", "", regex=False)
            .str.extract(r"(\d+(?:\.\d+)?)", expand=False)
        )
        cleaned["cost_for_two"] = pd.to_numeric(cleaned["cost_for_two"], errors="coerce")

    if "votes" in cleaned.columns:
        cleaned["votes"] = pd.to_numeric(cleaned["votes"], errors="coerce")

    for col in ("online_order", "book_table"):
        if col in cleaned.columns:
            normalized = cleaned[col].astype("string").str.strip().str.lower()
            cleaned[col] = normalized.map({
                "yes": 1, "no": 0, "true": 1, "false": 0, "1": 1, "0": 0
            }).where(normalized.notna(), np.nan)

    # Trim text fields used in grouping and analysis
    for col in ("name", "location", "cuisines", "address"):
        if col in cleaned.columns:
            cleaned[col] = cleaned[col].astype("string").str.strip()
            cleaned[col] = cleaned[col].replace({"": pd.NA, "nan": pd.NA})

    print(f"Rows after duplicate removal and type cleaning: {len(cleaned):,}")
    print("\nMissing values after cleaning (top 20):")
    display(cleaned.isna().sum().sort_values(ascending=False).head(20).rename("missing_count").to_frame())
    return cleaned


df_clean = clean_data(df)
print("\nCleaned shape:", df_clean.shape)
display(df_clean.head())


In [ ]:
# Cell 5 — Save the cleaned dataset
cleaned_path = OUTPUT_DIR / "cleaned_zomato.csv"
df_clean.to_csv(cleaned_path, index=False)
print(f"Cleaned dataset saved to: {cleaned_path.resolve()}")


In [ ]:
# Cell 6 — Basic statistics
print("=" * 60)
print("ZOMATO DATASET SUMMARY")
print("=" * 60)
print(f"Raw rows: {len(df):,}")
print(f"Cleaned rows: {len(df_clean):,}")
print(f"Columns after cleaning: {df_clean.shape[1]}")

if "name" in df_clean.columns:
    print(f"Unique restaurant names: {df_clean['name'].nunique(dropna=True):,}")
if "rate" in df_clean.columns and df_clean["rate"].notna().any():
    print(f"Average rating: {df_clean['rate'].mean():.2f}")
    print(f"Median rating: {df_clean['rate'].median():.2f}")
    print(f"Rating range: {df_clean['rate'].min():.1f}–{df_clean['rate'].max():.1f}")
if "cost_for_two" in df_clean.columns and df_clean["cost_for_two"].notna().any():
    print(f"Average cost for two: ₹{df_clean['cost_for_two'].mean():,.0f}")
    print(f"Median cost for two: ₹{df_clean['cost_for_two'].median():,.0f}")
if "votes" in df_clean.columns and df_clean["votes"].notna().any():
    print(f"Total votes: {df_clean['votes'].sum():,.0f}")
    print(f"Median votes: {df_clean['votes'].median():,.0f}")
for col, label in [("online_order", "Online-order availability"), ("book_table", "Table-booking availability")]:
    if col in df_clean.columns and df_clean[col].notna().any():
        print(f"{label}: {df_clean[col].mean() * 100:.1f}% (among non-missing values)")


In [ ]:
# Cell 7 — Rating distribution
if "rate" in df_clean.columns and df_clean["rate"].notna().any():
    plt.figure()
    sns.histplot(df_clean["rate"].dropna(), bins=20, kde=True)
    plt.title("Distribution of Restaurant Ratings")
    plt.xlabel("Rating")
    plt.ylabel("Number of records")
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "rating_distribution.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Rating chart skipped: no usable 'rate' column.")


In [ ]:
# Cell 8 — Top locations by restaurant count
if "location" in df_clean.columns:
    top_locations = df_clean["location"].value_counts(dropna=True).head(15)
    display(top_locations.rename("restaurant_records").to_frame())
    if not top_locations.empty:
        plt.figure()
        sns.barplot(x=top_locations.values, y=top_locations.index)
        plt.title("Top 15 Locations by Restaurant Records")
        plt.xlabel("Number of records")
        plt.ylabel("Location")
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / "top_locations.png", dpi=150, bbox_inches="tight")
        plt.show()
else:
    print("Location analysis skipped: no recognized location column.")


In [ ]:
# Cell 9 — Highest-rated locations, requiring at least 20 rated records
if {"location", "rate"}.issubset(df_clean.columns):
    location_ratings = (
        df_clean.dropna(subset=["location", "rate"])
        .groupby("location")
        .agg(avg_rating=("rate", "mean"), rated_records=("rate", "size"))
        .query("rated_records >= 20")
        .sort_values(["avg_rating", "rated_records"], ascending=[False, False])
        .head(10)
    )
    display(location_ratings.round(2))
    if not location_ratings.empty:
        plt.figure()
        sns.barplot(data=location_ratings.reset_index(), x="avg_rating", y="location")
        plt.title("Top Locations by Average Rating (at least 20 rated records)")
        plt.xlabel("Average rating")
        plt.ylabel("Location")
        plt.xlim(0, 5)
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / "top_locations_by_rating.png", dpi=150, bbox_inches="tight")
        plt.show()
else:
    print("Location rating analysis skipped: requires 'location' and 'rate'.")


In [ ]:
# Cell 10 — Most common cuisines (split comma-separated cuisine strings)
if "cuisines" in df_clean.columns:
    cuisine_counts = (
        df_clean["cuisines"].dropna()
        .str.split(",")
        .explode()
        .str.strip()
    )
    cuisine_counts = cuisine_counts[cuisine_counts.ne("")].value_counts().head(15)
    display(cuisine_counts.rename("record_count").to_frame())
    if not cuisine_counts.empty:
        plt.figure()
        sns.barplot(x=cuisine_counts.values, y=cuisine_counts.index)
        plt.title("Top 15 Cuisine Labels")
        plt.xlabel("Count across restaurant records")
        plt.ylabel("Cuisine")
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / "top_cuisines.png", dpi=150, bbox_inches="tight")
        plt.show()
else:
    print("Cuisine analysis skipped: no recognized cuisine column.")


In [ ]:
# Cell 11 — Cost distribution
if "cost_for_two" in df_clean.columns and df_clean["cost_for_two"].notna().any():
    costs = df_clean.loc[df_clean["cost_for_two"] >= 0, "cost_for_two"].dropna()
    if not costs.empty:
        plt.figure()
        sns.histplot(costs, bins=30)
        plt.title("Distribution of Approximate Cost for Two")
        plt.xlabel("Cost for two (dataset currency)")
        plt.ylabel("Number of records")
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / "cost_distribution.png", dpi=150, bbox_inches="tight")
        plt.show()
else:
    print("Cost chart skipped: no usable cost column.")


In [ ]:
# Cell 12 — Online order vs average rating
if {"online_order", "rate"}.issubset(df_clean.columns):
    order_rating = (
        df_clean.dropna(subset=["online_order", "rate"])
        .groupby("online_order")
        .agg(avg_rating=("rate", "mean"), records=("rate", "size"))
        .reset_index()
    )
    order_rating["online_order"] = order_rating["online_order"].map({1: "Yes", 0: "No"}).fillna(order_rating["online_order"].astype(str))
    display(order_rating.round(3))
    if not order_rating.empty:
        plt.figure()
        sns.barplot(data=order_rating, x="online_order", y="avg_rating")
        plt.title("Average Rating by Online-Order Availability")
        plt.xlabel("Online order available")
        plt.ylabel("Average rating")
        plt.ylim(0, 5)
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / "online_order_vs_rating.png", dpi=150, bbox_inches="tight")
        plt.show()
else:
    print("Online-order comparison skipped: requires online_order and rate columns.")


In [ ]:
# Cell 13 — Table booking vs average rating
if {"book_table", "rate"}.issubset(df_clean.columns):
    booking_rating = (
        df_clean.dropna(subset=["book_table", "rate"])
        .groupby("book_table")
        .agg(avg_rating=("rate", "mean"), records=("rate", "size"))
        .reset_index()
    )
    booking_rating["book_table"] = booking_rating["book_table"].map({1: "Yes", 0: "No"}).fillna(booking_rating["book_table"].astype(str))
    display(booking_rating.round(3))
    if not booking_rating.empty:
        plt.figure()
        sns.barplot(data=booking_rating, x="book_table", y="avg_rating")
        plt.title("Average Rating by Table-Booking Availability")
        plt.xlabel("Table booking available")
        plt.ylabel("Average rating")
        plt.ylim(0, 5)
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / "table_booking_vs_rating.png", dpi=150, bbox_inches="tight")
        plt.show()
else:
    print("Table-booking comparison skipped: requires book_table and rate columns.")


In [ ]:
# Cell 14 — Correlation heatmap for numeric columns
numeric_df = df_clean.select_dtypes(include="number")
if numeric_df.shape[1] >= 2:
    corr = numeric_df.corr(numeric_only=True)
    plt.figure(figsize=(10, 7))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
    plt.title("Correlation Heatmap (Numeric Columns)")
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "correlation_heatmap.png", dpi=150, bbox_inches="tight")
    plt.show()
    display(corr.round(3))
else:
    print("Correlation heatmap skipped: fewer than two numeric columns.")


In [ ]:
# Cell 15 — Top restaurants by votes (if name and votes exist)
if {"name", "votes"}.issubset(df_clean.columns):
    top_voted = df_clean.dropna(subset=["name", "votes"]).sort_values("votes", ascending=False).head(10)
    display(top_voted[[c for c in ["name", "location", "rate", "votes", "cost_for_two"] if c in top_voted.columns]])
else:
    print("Top-voted restaurant table skipped: requires name and votes columns.")


In [ ]:
# Cell 16 — Top-rated restaurants with a minimum vote threshold (reduces tiny-sample rankings)
if {"name", "rate", "votes"}.issubset(df_clean.columns):
    rated = df_clean.dropna(subset=["name", "rate", "votes"]).copy()
    top_rated = rated[rated["votes"] >= 50].sort_values(["rate", "votes"], ascending=[False, False]).head(10)
    display(top_rated[[c for c in ["name", "location", "rate", "votes", "cost_for_two"] if c in top_rated.columns]])
else:
    print("Top-rated restaurant table skipped: requires name, rate and votes columns.")


In [ ]:
# Cell 17 — Rating categories on a copy (no side effects on df_clean)
if "rate" in df_clean.columns:
    rating_view = df_clean.copy()
    rating_view["rating_category"] = pd.cut(
        rating_view["rate"],
        bins=[-0.001, 2.0, 3.0, 4.0, 5.0],
        labels=["Low (0–2)", "Average (2–3)", "Good (3–4)", "Excellent (4–5)"],
        include_lowest=True
    )
    category_counts = rating_view["rating_category"].value_counts(sort=False, dropna=False)
    display(category_counts.rename("record_count").to_frame())
    plot_counts = category_counts.dropna()
    if not plot_counts.empty:
        plt.figure()
        sns.barplot(x=plot_counts.index.astype(str), y=plot_counts.values)
        plt.title("Restaurant Records by Rating Category")
        plt.xlabel("Rating category")
        plt.ylabel("Number of records")
        plt.xticks(rotation=15)
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / "rating_categories.png", dpi=150, bbox_inches="tight")
        plt.show()
else:
    print("Rating categories skipped: no usable 'rate' column.")


In [ ]:
# Cell 18 — Data quality summary and output files
quality_report = pd.DataFrame({
    "raw_missing": df.isna().sum(),
    "cleaned_missing": df_clean.reindex(columns=df.columns).isna().sum()
})
display(quality_report.sort_values("raw_missing", ascending=False).head(20))

print(f"Raw dataset: {len(df):,} rows × {df.shape[1]} columns")
print(f"Cleaned dataset: {len(df_clean):,} rows × {df_clean.shape[1]} columns")
print(f"Plots folder: {PLOTS_DIR.resolve()}")
print(f"Cleaned CSV: {cleaned_path.resolve()}")

print("\nNotebook run complete. Use the actual output values when writing project conclusions.")
